In [2]:
import os
import time
import json

import openai
from openai import OpenAI

from elasticsearch import Elasticsearch

import spacy
import numpy as np

import pandas as pd
from tqdm.auto import tqdm

In [3]:
df_ground_truth = pd.read_csv('../data/ground_truth_NEW.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

In [4]:
ground_truth[0]

{'question': 'How do I sign up for an account on your website?',
 'document': 'doc_0_how_can_i_create_an_account_'}

In [5]:
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [10]:
es_client = Elasticsearch('http://localhost:9200')
es_client

<Elasticsearch(['http://localhost:9200'])>

In [14]:
index_name='documents'
try:
    result = es_client.count(index=index_name)
    print(f"ES Checking = Document count in {index_name}: {result['count']}")
except Exception as e:
    print(f"ES Checking = Error: {str(e)}")

ES Checking = Document count in documents: 79


In [16]:
q = ground_truth[0]['question']
d = ground_truth[0]['document']
q, d

('How do I sign up for an account on your website?',
 'doc_0_how_can_i_create_an_account_')

In [17]:
def elastic_search_text(query, index_name="documents"):
    search_query = {
        "size": 10,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "answer"],
                        "type": "best_fields"
                    }
                },
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)
    return [hit["_source"] for hit in response["hits"]["hits"]]

In [18]:
text_results = elastic_search_text(ground_truth[0]['question'])
text_results = [item['document_id'] for item in text_results]
text_results

['doc_0_how_can_i_create_an_account_',
 'doc_64_can_i_request_a_product_if_it_',
 'doc_16_can_i_order_without_creating_a',
 'doc_30_do_you_offer_installation_serv',
 'doc_62_can_i_order_a_product_if_it_is',
 'doc_37_can_i_request_an_invoice_for_m',
 'doc_2_how_can_i_track_my_order_',
 'doc_9_how_can_i_contact_customer_sup',
 'doc_5_how_long_does_shipping_take_',
 'doc_19_how_can_i_leave_a_product_revi']

In [19]:
import spacy
import numpy as np

nlp = spacy.load('en_core_web_sm')

In [20]:
def get_vector(query):
    doc = nlp(query)
    tokens = [token.lemma_ for token in doc]
    text = ' '.join(tokens)
    doc_lemmatized = nlp(text)
    vector = np.mean([token.vector for token in doc_lemmatized], axis=0).tolist()
    return vector

In [21]:
def elastic_search_knn(query, index_name="documents", field='embedding'):
                
    vector = get_vector(query)

    search_body = {
        "knn": {
            "field": "embedding",
            "query_vector": vector,
            "k": 10,
            "num_candidates": 100
        },
        "size": 10,
        "_source": ['document_id', 'question', 'answer'],
    }

    es_results = es_client.search(index=index_name, body=search_body)

    return [hit["_source"] for hit in es_results["hits"]["hits"]]

In [22]:
knn_results = elastic_search_knn(ground_truth[0]['question'])
knn_results = [item['document_id'] for item in knn_results]
knn_results

['doc_21_what_should_i_do_if_i_receive_',
 'doc_0_how_can_i_create_an_account_',
 'doc_40_can_i_request_a_product_that_i',
 'doc_64_can_i_request_a_product_if_it_',
 'doc_70_can_i_request_a_product_that_i',
 'doc_3_what_is_your_return_policy_',
 'doc_8_can_i_change_my_shipping_addre',
 'doc_15_do_you_have_a_loyalty_program_',
 'doc_2_how_can_i_track_my_order_',
 'doc_18_can_i_change_or_cancel_an_item']

In [23]:
ground_truth[:3]

[{'question': 'How do I sign up for an account on your website?',
  'document': 'doc_0_how_can_i_create_an_account_'},
 {'question': 'Where can I find the Sign Up button to make a new account?',
  'document': 'doc_0_how_can_i_create_an_account_'},
 {'question': 'What steps do I need to follow to register a new account?',
  'document': 'doc_0_how_can_i_create_an_account_'}]

In [25]:
truth_question = df_ground_truth.iloc[0]['question']
truth_document_id = df_ground_truth.iloc[0]['document']
truth_question, truth_document_id

('How do I sign up for an account on your website?',
 'doc_0_how_can_i_create_an_account_')

In [26]:
def hit_rate_one(original_id, search_results):
    return 1 if original_id in search_results else 0

In [27]:
def mrr_one(original_id, search_results):
    mrr = 0
    for position in range(len(search_results)):
        if search_results[position] == original_id:
            mrr += 1 / (position + 1)
    return mrr

In [28]:
text_results = elastic_search_text(truth_question)
text_results = [item['document_id'] for item in text_results]
text_results, hit_rate_one(truth_document_id, text_results)

(['doc_0_how_can_i_create_an_account_',
  'doc_64_can_i_request_a_product_if_it_',
  'doc_16_can_i_order_without_creating_a',
  'doc_30_do_you_offer_installation_serv',
  'doc_62_can_i_order_a_product_if_it_is',
  'doc_37_can_i_request_an_invoice_for_m',
  'doc_2_how_can_i_track_my_order_',
  'doc_9_how_can_i_contact_customer_sup',
  'doc_5_how_long_does_shipping_take_',
  'doc_19_how_can_i_leave_a_product_revi'],
 1)

In [29]:
knn_results = elastic_search_knn(truth_question)

knn_results = [item['document_id'] for item in knn_results]
knn_results, hit_rate_one(truth_document_id, knn_results)

(['doc_21_what_should_i_do_if_i_receive_',
  'doc_0_how_can_i_create_an_account_',
  'doc_40_can_i_request_a_product_that_i',
  'doc_64_can_i_request_a_product_if_it_',
  'doc_70_can_i_request_a_product_that_i',
  'doc_3_what_is_your_return_policy_',
  'doc_8_can_i_change_my_shipping_addre',
  'doc_15_do_you_have_a_loyalty_program_',
  'doc_2_how_can_i_track_my_order_',
  'doc_18_can_i_change_or_cancel_an_item'],
 1)

In [30]:
def elastic_search_knn_combined_style(query, index_name="documents", field='embedding'):
    # Obtain the vector representation of the query
    vector = get_vector(query)

    # Construct the search query using a script score for cosine similarity
    search_query = {
        "size": 10,  # Number of results to return
        "query": {
            "bool": {
                "must": [
                    {
                        "script_score": {
                            "query": {
                                "match_all": {}  # Match all documents to apply custom scoring
                            },
                            "script": {
                                "source": """
                                    cosineSimilarity(params.query_vector, 'embedding') + 1
                                """,  # +1 to ensure the score is positive
                                "params": {
                                    "query_vector": vector
                                }
                            }
                        }
                    }
                ]
            }
        },
        "_source": ['document_id', 'question', 'answer']  # Fields to return in the results
    }

    # Perform the search with the constructed query
    es_results = es_client.search(index=index_name, body=search_query)

    # Extract and return the results
    result_docs = [hit["_source"] for hit in es_results["hits"]["hits"]]

    return result_docs

In [31]:
knn_results = elastic_search_knn_combined_style(truth_question)
knn_combined_results = [item['document_id'] for item in knn_results]
knn_combined_results, hit_rate_one(truth_document_id, knn_combined_results)

(['doc_21_what_should_i_do_if_i_receive_',
  'doc_0_how_can_i_create_an_account_',
  'doc_40_can_i_request_a_product_that_i',
  'doc_64_can_i_request_a_product_if_it_',
  'doc_3_what_is_your_return_policy_',
  'doc_70_can_i_request_a_product_that_i',
  'doc_8_can_i_change_my_shipping_addre',
  'doc_15_do_you_have_a_loyalty_program_',
  'doc_18_can_i_change_or_cancel_an_item',
  'doc_2_how_can_i_track_my_order_'],
 1)

In [32]:
truth_question

'How do I sign up for an account on your website?'

In [33]:
df_ground_truth

,question,document
0,How do I sign up for an account on your website?,doc_0_how_can_i_create_an_account_
1,Where can I find the Sign Up button to make a ...,doc_0_how_can_i_create_an_account_
2,What steps do I need to follow to register a n...,doc_0_how_can_i_create_an_account_
3,Can you tell me how to create an account on th...,doc_0_how_can_i_create_an_account_
4,How do I complete the registration process aft...,doc_0_how_can_i_create_an_account_
...,...,...
390,"If I bought something during a promo sale, can...",doc_78_can_i_return_a_product_if_it_w
391,Will I get refunded for the discounted price i...,doc_78_can_i_return_a_product_if_it_w
392,Are returns allowed for products purchased on ...,doc_78_can_i_return_a_product_if_it_w
393,"If I return an item I got with a discount, how...",doc_78_can_i_return_a_product_if_it_w


In [35]:
hit_rate_results_text = []
hit_rate_results_vector = []
hit_rate_results_vector_combined = []
mrr_results_text = []
mrr_results_vector = []
mrr_results_vector_combined = []

for index, row in tqdm(df_ground_truth.iterrows(), total=df_ground_truth.shape[0], desc="Processing rows"):
    document_id = row['document']
    question = row['question']
    
    text_results = elastic_search_text(question)
    text_results = [item['document_id'] for item in text_results]
    
    hit_rate_text = hit_rate_one(document_id, text_results)
    hit_rate_results_text.append(hit_rate_text)
    mrr_text = mrr_one(document_id, text_results)
    mrr_results_text.append(mrr_text)

    knn_results = elastic_search_knn(question)
    knn_results = [item['document_id'] for item in knn_results]

    hit_rate_vector = hit_rate_one(document_id, knn_results)
    hit_rate_results_vector.append(hit_rate_vector)
    mrr_vector = mrr_one(document_id, knn_results)
    mrr_results_vector.append(mrr_vector)

    knn_combined_results = elastic_search_knn_combined_style(truth_question)
    knn_combined_results = [item['document_id'] for item in knn_combined_results]
    hit_rate_vector_combined = hit_rate_one(document_id, knn_combined_results)
    hit_rate_results_vector_combined.append(hit_rate_vector_combined)
    mrr_vector_combined = mrr_one(document_id, knn_combined_results)
    mrr_results_vector_combined.append(mrr_vector_combined)

Processing rows:   0%|          | 0/395 [00:00<?, ?it/s]

In [36]:
len(df_ground_truth), len(hit_rate_results_text), len(hit_rate_results_vector), len(mrr_results_text), len(mrr_results_vector), len(hit_rate_results_vector_combined), len(mrr_results_vector_combined)

(395, 395, 395, 395, 395, 395, 395)

In [37]:
df_ground_truth['hit_rate_text'] = hit_rate_results_text
df_ground_truth['hit_rate_vector'] = hit_rate_results_vector
df_ground_truth['hit_rate_vector_combined'] = hit_rate_results_vector_combined
df_ground_truth['mrr_text'] = mrr_results_text
df_ground_truth['mrr_vector'] = mrr_results_vector
df_ground_truth['mrr_vector_combined'] = mrr_results_vector_combined

In [38]:
df_ground_truth

,question,document,hit_rate_text,hit_rate_vector,hit_rate_vector_combined,mrr_text,mrr_vector,mrr_vector_combined
0,How do I sign up for an account on your website?,doc_0_how_can_i_create_an_account_,1,1,1,1.000000,0.50,0.5
1,Where can I find the Sign Up button to make a ...,doc_0_how_can_i_create_an_account_,1,1,1,1.000000,0.25,0.5
2,What steps do I need to follow to register a n...,doc_0_how_can_i_create_an_account_,1,1,1,0.200000,0.10,0.5
3,Can you tell me how to create an account on th...,doc_0_how_can_i_create_an_account_,1,1,1,1.000000,0.50,0.5
4,How do I complete the registration process aft...,doc_0_how_can_i_create_an_account_,1,0,1,0.500000,0.00,0.5
...,...,...,...,...,...,...,...,...
390,"If I bought something during a promo sale, can...",doc_78_can_i_return_a_product_if_it_w,1,0,0,0.200000,0.00,0.0
391,Will I get refunded for the discounted price i...,doc_78_can_i_return_a_product_if_it_w,1,1,0,0.500000,1.00,0.0
392,Are returns allowed for products purchased on ...,doc_78_can_i_return_a_product_if_it_w,0,0,0,0.000000,0.00,0.0
393,"If I return an item I got with a discount, how...",doc_78_can_i_return_a_product_if_it_w,0,0,0,0.000000,0.00,0.0


In [39]:
df_ground_truth.describe()

,hit_rate_text,hit_rate_vector,hit_rate_vector_combined,mrr_text,mrr_vector,mrr_vector_combined
count,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
mean,0.870886,0.498734,0.126582,0.596194,0.223967,0.037076
std,0.335751,0.500633,0.332926,0.393669,0.329718,0.135237
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.250000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.500000,0.000000,0.000000
75%,1.000000,1.000000,0.000000,1.000000,0.333333,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [40]:
df_ground_truth.to_csv('../data/ground_truth_metrics_retrieval.csv', index=False, sep=';', encoding='utf-8')